# Direct vs Indirect Standardization: What the Method Cannot Fix

*Demographic Analysis Series · Module 1 (Mortality) · Lesson 1.3b*
*Author: Jiajia Li | Published: November 7, 2026 | Reading time: about 25 minutes*

[Lesson 1.2b](/blog/demographic-m1l2b-standardization-worked-example) standardized directly. [Lesson 1.3a](/blog/demographic-m1l3a-indirect-standardization-smr) standardized indirectly, through the standardized mortality ratio. Each lesson ended with a warning: the direct route collapses when age-specific death counts are too thin, and the indirect route hides a standard inside the study population's own age structure. This lesson puts the two routes on the same data, shows that they are one operation with different weights, and then maps what the whole standardization enterprise cannot fix.

By the end of this notebook you will be able to:

- Reproduce both estimates for Malaysia and Australia in 1988 and prove the identity that links them
- Derive any standardized ratio as a deaths-weighted average of age-specific rate ratios
- Explain why five full-population standards agree here while a change of age scope moves the answer from 1.41 to 2.01
- Show that two populations with identical age-specific rates can receive different SMRs purely because their age structures differ
- Decide when to stop summarizing and report age-specific rates directly

## 1. Two Routes to One Comparison

Direct standardization applies one set of common age weights to every population's own age-specific mortality schedule. Indirect standardization applies one trusted mortality schedule to every population's age structure and compares observed with expected deaths. The inputs look so different that the two methods are usually taught as alternatives. The 1988 Malaysia–Australia data will show that they are the same operation.

The data are the same as in Lessons 1.1c through 1.3a: Carmichael's Table 1.2 (age-specific death rates per 1,000) and Table 1.3 (population by age and sex per million) for Malaysia and Australia in 1988. Deaths are reconstructed as rate times population, so totals inherit the rounding of the printed tables. The code keeps the printed male and female columns and aggregates them. The next two cells rebuild the tables and compute the direct and indirect estimates.

In [ ]:
# ---- 1. Load Tables 1.2/1.3 (same embedded data as Lessons 1.1c-1.3a) ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from io import StringIO
from IPython.display import display

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial'],
    'font.size': 11,
    'figure.dpi': 125,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.8,
    'legend.frameon': False,
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
})
GREEN, RUST = '#35634e', '#a95b3d'

# Carmichael (2016), Ch. 1: Table 1.2 (rates per 1,000) and Table 1.3 (persons per 1,000,000), 1988.
rates_csv = """
age,m_malaysia,m_australia,f_malaysia,f_australia
0,16.6,9.8,12.8,7.6
1-4,1.7,0.5,1.1,0.4
5-9,0.6,0.2,0.5,0.2
10-14,0.6,0.3,0.4,0.2
15-19,1.0,1.1,0.5,0.4
20-24,1.5,1.6,0.6,0.5
25-29,1.7,1.5,0.9,0.5
30-34,2.0,1.4,1.1,0.6
35-39,2.4,1.5,1.5,0.8
40-44,3.4,2.2,2.3,1.2
45-49,5.5,3.4,3.2,2.1
50-54,9.9,6.0,5.9,3.4
55-59,15.4,10.0,9.9,5.5
60-64,25.5,17.3,17.0,8.7
65-69,38.3,27.2,27.8,13.8
70-74,62.7,45.3,46.6,23.5
75-79,86.2,71.9,69.8,40.7
80-84,144.3,110.7,119.0,71.4
85+,173.4,186.6,142.8,147.7
"""
pop_csv = """
age,m_malaysia,m_australia,f_malaysia,f_australia
0,14254,7597,13464,7245
1-4,56773,30324,53804,28954
5-9,61207,37762,58012,35803
10-14,56002,38712,53613,36758
15-19,52776,43626,50772,41743
20-24,50193,40771,49060,39297
25-29,42748,42914,44839,41982
30-34,35704,40234,38326,39933
35-39,30427,38778,31269,38391
40-44,23858,36248,23075,34537
45-49,20473,27923,19767,26357
50-54,17052,23855,17190,22798
55-59,12615,22708,13604,21924
60-64,10043,21764,10513,22336
65-69,7373,17701,8498,19973
70-74,4608,12857,5246,16200
75-79,3430,8712,3976,12555
80-84,1399,4471,1675,7827
85+,963,2319,1399,6111
"""
rates = pd.read_csv(StringIO(rates_csv), index_col='age')
pop = pd.read_csv(StringIO(pop_csv), index_col='age')
deaths = pop * rates / 1000

exposure = pd.DataFrame({
    'Malaysia': pop['m_malaysia'] + pop['f_malaysia'],
    'Australia': pop['m_australia'] + pop['f_australia'],
})
structure = exposure / exposure.sum()
asdr = pd.DataFrame({
    'Malaysia': (deaths['m_malaysia'] + deaths['f_malaysia']) / exposure['Malaysia'] * 1000,
    'Australia': (deaths['m_australia'] + deaths['f_australia']) / exposure['Australia'] * 1000,
})
total_deaths = pd.Series({
    'Malaysia': deaths[['m_malaysia', 'f_malaysia']].sum().sum(),
    'Australia': deaths[['m_australia', 'f_australia']].sum().sum(),
})
cdr = total_deaths / exposure.sum() * 1000

summary = pd.DataFrame({
    'Total deaths (per million pop.)': total_deaths.round(0),
    'CDR per 1,000': cdr.round(2),
})
display(summary)
assert np.allclose(cdr.to_numpy(), [4.952, 7.251], atol=1e-3)

# Age midpoints on the nineteen-group grid and on the five-year grid used for standards.
mid = pd.Series({a: (0.5 if a == '0' else 2.5 if a == '1-4' else 87.5 if a == '85+'
                     else int(a.split('-')[0]) + 2.5) for a in asdr.index})
five_yr = [f'{x}-{x+4}' for x in range(0, 85, 5)] + ['85+']


def to_five_year(values):
    """Aggregate ages 0 and 1-4 into 0-4, then select five-year groups."""
    out = values.copy()
    out.loc['0-4'] = out.loc['0'] + out.loc['1-4']
    return out.loc[five_yr]


deaths5 = pd.DataFrame({
    'Malaysia': to_five_year(deaths['m_malaysia'] + deaths['f_malaysia']),
    'Australia': to_five_year(deaths['m_australia'] + deaths['f_australia']),
})
exposure5 = pd.DataFrame({
    'Malaysia': to_five_year(exposure['Malaysia']),
    'Australia': to_five_year(exposure['Australia']),
})
asdr5 = deaths5 / exposure5 * 1000
mid5 = pd.Series({a: (2.5 if a == '0-4' else 87.5 if a == '85+'
                      else int(a.split('-')[0]) + 2.5) for a in five_yr})

# Segi and WHO weights, transcribed in Lesson 1.2a from Ahmad et al. (2001), Table 1.
segi = pd.Series([12, 10, 9, 9, 8, 8, 6, 6, 6, 6, 5, 4, 4, 3, 2, 1, 0.5, 0.5], index=five_yr)
segi = segi / segi.sum()
who = pd.Series([8.86, 8.69, 8.60, 8.47, 8.22, 7.93, 7.61, 7.15, 6.59,
                 6.04, 5.37, 4.55, 3.72, 2.96, 2.21, 1.52, 0.91, 0.63], index=five_yr)
who = who / who.sum()

In [ ]:
# ---- 2. Direct and indirect estimates on the same tables ----
def standardize_direct(rates, weights):
    """Direct standardized rate: sum of age-specific rates times standard weights.

    rates: Series of age-specific rates, or DataFrame with one column per population.
    weights: Series of standard-population shares on the same age index.
    Weights are normalized internally; the unit is inherited from `rates`.
    """
    if not rates.index.equals(weights.index):
        raise ValueError('rates and weights must share an identical age index')
    if (weights < 0).any():
        raise ValueError('standard weights must be nonnegative')
    w = weights / weights.sum()
    if isinstance(rates, pd.DataFrame):
        return rates.mul(w, axis='index').sum()
    return float((rates * w).sum())


def standardize_indirect(observed_deaths, population, std_rates, unit=1000):
    """SMR: observed deaths divided by deaths expected under a standard schedule.

    Returns (smr, expected_deaths); both belong in an honest report.
    """
    if not population.index.equals(std_rates.index):
        raise ValueError('population and standard rates must share an identical age index')
    if (std_rates < 0).any():
        raise ValueError('standard rates must be nonnegative')
    expected = float((population * std_rates / unit).sum())
    if expected <= 0:
        raise ValueError('expected deaths must be positive')
    return observed_deaths / expected, expected


# Toy checks carried over from Lessons 1.2b and 1.3a.
assert np.isclose(standardize_direct(
    pd.Series([1.0, 20.0], index=['young', 'old']),
    pd.Series([0.8, 0.2], index=['young', 'old'])), 4.8)
toy_smr, toy_e = standardize_indirect(
    45, pd.Series([1000, 1000], index=['young', 'old']),
    pd.Series([10.0, 20.0], index=['young', 'old']))
assert np.isclose(toy_smr, 1.5) and np.isclose(toy_e, 30)

# Direct standardized ratios on the two own structures.
w_aus, w_mal = structure['Australia'], structure['Malaysia']
ratio_on_aus = standardize_direct(asdr['Malaysia'], w_aus) / standardize_direct(asdr['Australia'], w_aus)
ratio_on_mal = standardize_direct(asdr['Malaysia'], w_mal) / standardize_direct(asdr['Australia'], w_mal)

# Indirect estimates in both directions.
smr_mal, e_mal = standardize_indirect(total_deaths['Malaysia'], exposure['Malaysia'], asdr['Australia'])
smr_aus, e_aus = standardize_indirect(total_deaths['Australia'], exposure['Australia'], asdr['Malaysia'])

print(f"Malaysia: {total_deaths['Malaysia']:,.0f} observed, {e_mal:,.0f} expected under Australia's schedule "
      f"-> SMR {smr_mal:.3f}")
print(f"Australia: {total_deaths['Australia']:,.0f} observed, {e_aus:,.0f} expected under Malaysia's schedule "
      f"-> SMR {smr_aus:.3f}")

estimates = pd.DataFrame([
    ('Crude rate ratio', "each population's own structure", cdr['Malaysia'] / cdr['Australia']),
    ('Direct standardized ratio', 'Australian structure', ratio_on_aus),
    ('Direct standardized ratio', 'Malaysian structure', ratio_on_mal),
    ('SMR (indirect)', 'Australian schedule; Malaysian structure', smr_mal),
    ('SMR (indirect)', 'Malaysian schedule; Australian structure', smr_aus),
    ('Ratio of the two SMRs', 'two different implicit standards', smr_mal / smr_aus),
], columns=['Estimate', 'Standard / reference', 'Value'])
display(estimates.assign(Value=estimates['Value'].round(3)).set_index('Estimate'))

# Identities proved in the next section.
assert np.isclose(ratio_on_mal, smr_mal)
assert np.isclose(ratio_on_aus, 1 / smr_aus)
assert np.isclose(standardize_direct(asdr['Australia'], w_aus), cdr['Australia'])

### The identity that dissolves the distinction

Write out the SMR for study population $i$ against standard schedule $s$:

$$
\mathrm{SMR}_i(s)
= \frac{O_i}{\sum_x m_s(x)\,P_i(x)}
= \frac{\sum_x m_i(x)\,P_i(x)}{\sum_x m_s(x)\,P_i(x)}
= \frac{\sum_x p_i(x)\,m_i(x)}{\sum_x p_i(x)\,m_s(x)}
= \frac{M_i^{(i)}}{M_s^{(i)}}.
$$

Write $M_g^{(s)}=\sum_x p_s(x)\,m_g(x)$ for the direct standardized rate of schedule $g$ under structure $s$. The second expression uses $O_i=\sum_x m_i(x)P_i(x)$; the third writes $p_i(x)=P_i(x)/\sum_yP_i(y)$ for $i$'s age structure. The last expression is therefore the direct standardized ratio computed on $i$'s own structure. In one sentence:

> The indirect method is the direct method with the standard population set to the study population itself.

The code above checks both directions. Malaysia's SMR against the Australian schedule equals the direct ratio on Malaysia's structure (1.517); Australia's SMR against the Malaysian schedule equals the reciprocal of the direct ratio on Australia's structure ($1/1.4586=0.686$).

This resolves the question Lesson 1.3a left open. The routes do not disagree about a number; they answer different questions with the same algebra. The direct route lets the analyst choose the standard and watch the estimate move. The indirect route fixes the standard at the study population's own structure — and then conceals the choice inside the ratio. Lesson 1.3a's warning about the quotient of two SMRs now has a one-line proof: $1.517/0.686=2.21$ compares Malaysia-on-Malaysia with Australia-on-Australia, not Malaysia with Australia.

## 2. Every Standardized Ratio Is a Deaths-Weighted Average

The identity generalizes. For a standard population with shares $w_s(x)$ and a study schedule $m_i(x)$, define the age-specific rate ratio $r(x)=m_i(x)/m_s(x)$. Then

$$
R_s(i)=\frac{\sum_x w_s(x)\,m_i(x)}{\sum_x w_s(x)\,m_s(x)}
=\sum_x q_s(x)\,r(x),
\qquad
q_s(x)=\frac{w_s(x)\,m_s(x)}{\sum_y w_s(y)\,m_s(y)}.
$$

The weights $q_s$ are not the standard's population shares. They are the deaths the standard population would experience under the schedule $m_s$ — its own expected death distribution. The equation carries the whole mechanism:

- $r(x)$ carries the substantive comparison: how much more, or less, the study population dies at each age.
- $q_s(x)$ carries every methodological choice: which ages the standard emphasizes, net of how dangerous those ages are.
- The standardized ratio is the weighted average of the first by the second.

When $r(x)$ is constant across ages, every $q_s$ returns that constant — the proportional-schedule theorem of Lesson 1.2a. When $r(x)$ varies, different defensible standards can return different answers. Standardization does not eliminate judgment; it relocates judgment into the choice of $q$.

Here the denominator schedule is Australia's. So $q_{\text{Australia}}$ is Australia's own expected death distribution, while $q_{\text{Malaysia}}$ applies Australia's schedule to Malaysia's much younger structure. (As in Lesson 1.3a, a ratio of the two countries' SMRs would mix the two $q$'s and answer no well-posed question.)

In [ ]:
# ---- 3. The ratio curve and the weights that average it ----
r_age = asdr['Malaysia'] / asdr['Australia']


def implied_death_weights(weights, schedule):
    """q(x): the standard population's expected death distribution under `schedule`."""
    if not weights.index.equals(schedule.index):
        raise ValueError('weights and schedule must share an identical age index')
    q = weights * schedule
    return q / q.sum()


q_aus = implied_death_weights(structure['Australia'], asdr['Australia'])
q_mal = implied_death_weights(structure['Malaysia'], asdr['Australia'])
assert np.isclose(q_aus.sum(), 1) and np.isclose(q_mal.sum(), 1)

# The same dot product with two different weight distributions.
assert np.isclose(float((q_aus * r_age).sum()), ratio_on_aus)
assert np.isclose(float((q_mal * r_age).sum()), ratio_on_mal)

print(f"Expected deaths at 65+: {q_aus[mid >= 65].sum():.0%} with Australia's structure, "
      f"{q_mal[mid >= 65].sum():.0%} with Malaysia's")

# Five full-population standards, evaluated on the common five-year grid.
five_standards = {
    'Australia': exposure5['Australia'] / exposure5['Australia'].sum(),
    'Malaysia': exposure5['Malaysia'] / exposure5['Malaysia'].sum(),
    'Segi': segi,
    'WHO': who,
    'Pooled': (exposure5['Australia'] + exposure5['Malaysia'])
              / (exposure5['Australia'] + exposure5['Malaysia']).sum(),
}
full_ratio = pd.Series({
    name: float((w * asdr5['Malaysia']).sum() / (w * asdr5['Australia']).sum())
    for name, w in five_standards.items()})
q_mean_age = pd.Series({
    name: float(((w * asdr5['Australia']) / (w * asdr5['Australia']).sum() * mid5).sum())
    for name, w in five_standards.items()})
display(pd.DataFrame({'Malaysia ÷ Australia': full_ratio.round(3),
                      'Mean age of expected deaths': q_mean_age.round(1)}))
assert full_ratio.between(1.45, 1.53).all()

# Figure 1: the ratio curve (left) and the weights that average it (right).
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.6, 4.0), layout='constrained')

x = np.arange(len(r_age))
ax1.plot(x, r_age, color=GREEN, marker='o', markersize=4, linewidth=1.6)
ax1.axhline(1.0, color='#7b8078', linewidth=1, linestyle='--')
ax1.set_xticks(x)
ax1.set_xticklabels(r_age.index, rotation=45, ha='right')
ax1.set_xlim(-0.6, 19.0)
ax1.set_xlabel('Age group')
ax1.set_ylabel('Mortality ratio, Malaysia ÷ Australia')
ax1.set_title('Malaysia is worse at every age except 85+', loc='left', pad=14)
ax1.set_axisbelow(True)
ax1.grid(axis='y', color='#e6e8e3', linewidth=0.8)
ax1.annotate('3.12', (1, 3.12), textcoords='offset points', xytext=(0, 7),
             ha='center', fontsize=8.5, color=GREEN)
ax1.annotate('0.98', (18, 0.98), textcoords='offset points', xytext=(12, -8),
             ha='left', fontsize=8.5, color=RUST)

width = 0.4
ax2.bar(x - width / 2, q_aus * 100, width, color=GREEN, label='Australian structure')
ax2.bar(x + width / 2, q_mal * 100, width, color=RUST, alpha=0.8, label='Malaysian structure')
ax2.set_xticks(x)
ax2.set_xticklabels(r_age.index, rotation=45, ha='right')
ax2.set_xlim(-0.7, 18.7)
ax2.set_xlabel('Age group')
ax2.set_ylabel('Share of expected deaths (%)')
ax2.legend(loc='upper left', fontsize=9)
ax2.set_title("Where each structure's expected deaths fall", loc='left', pad=14)
ax2.set_axisbelow(True)
ax2.grid(axis='y', color='#e6e8e3', linewidth=0.8)
plt.show()

**Figure 1 | The ratio curve and the weights that average it.** Left: Malaysia's mortality relative to Australia's in each age group (Carmichael Tables 1.2 and 1.3, both sexes combined). The ratio exceeds 1 at every age except 85+, peaking at 3.12 at ages 1–4. Right: the expected-death weights $q(x)$ that each structure implies under the Australian schedule. Australia's older structure places 72% of its expected deaths at ages 65 and over; Malaysia's younger structure places 51%. The standardized ratio is the left panel weighted by the right: 1.46 with the Australian weights, 1.52 with the Malaysian ones. Both ends of the ratio curve are far from 1.5 — 3.12 among toddlers, 0.98 at 85+ — but almost all of the weight sits between ages 25 and 84, where the ratio is roughly 1.45 to 1.7. That is why the five full-population standards in the table agree to within 0.06.

## 3. The Standard Is a Choice of Weights

The table above puts five full-population standards on the same five-year grid: each country's own structure, the Segi world standard, the WHO World Standard, and the pooled Malaysia-plus-Australia population. Their answers lie between 1.46 and 1.52, and their implied expected-death mean ages span only about twelve years, from 58 for Malaysia's young structure to 70 for Australia's old one. For this comparison the choice among full-population standards barely matters. That is a property of these data, not a theorem: the ratio curve is flat across the ages where every standard places its deaths.

But "all ages" is itself a choice, and often not the question. A ministry asking about child mortality is not asking the same question as one asking about mortality at older ages. Standardization can answer either, if the standard's weights are restricted to the relevant range and renormalized inside it:

$$
R_s^{(A)}(i)=\frac{\sum_{x\in A}\tilde w_s(x)\,m_i(x)}{\sum_{x\in A}\tilde w_s(x)\,m_s(x)},
\qquad
\tilde w_s(x)=\frac{w_s(x)}{\sum_{y\in A}w_s(y)}.
$$

With the WHO standard, the answer is 2.01 for ages 0–14, 1.56 for ages 15–64, 1.41 for ages 65 and over, and 1.47 for all ages. The age scope moves the answer far more than the choice among full-population standards. The full range is wider still: because every standardized ratio is a weighted average of $r(x)$, it cannot leave the interval spanned by the age-specific ratios — from 0.98 at 85+ to 2.76 on the five-year grid, or 3.12 if ages 1–4 are separated.

So standardization does not produce "the" comparative mortality ratio. It produces the answer to a counterfactual question: *what if the ages counted like this?* Different reasonable answers to "this" give different numbers. An honest report names the weights and, when the conclusion is not robust, gives the range.

In [ ]:
# ---- 4. Full-population standards versus age scopes ----
print(f"Five full-population standards span {full_ratio.min():.3f} to {full_ratio.max():.3f}; "
      f"their expected deaths have mean ages {q_mean_age.min():.1f} to {q_mean_age.max():.1f}.")

scopes = {
    'Ages 0–14': (0, 15),
    'Ages 15–64': (15, 65),
    'Ages 65+': (65, 120),
    'All ages': (0, 120),
}
scope_ratio = {}
for name, (lo, hi) in scopes.items():
    inside = (mid5 >= lo) & (mid5 < hi)
    w = who[inside] / who[inside].sum()          # WHO weights renormalized inside the scope
    scope_ratio[name] = float((w * asdr5['Malaysia'][inside]).sum()
                              / (w * asdr5['Australia'][inside]).sum())
scope_ratio = pd.Series(scope_ratio)
display(scope_ratio.round(3).to_frame('Malaysia ÷ Australia'))
assert np.isclose(scope_ratio['All ages'], full_ratio['WHO'])
assert scope_ratio['Ages 0–14'] > 1.9 and scope_ratio['Ages 65+'] < 1.45

r5 = asdr5['Malaysia'] / asdr5['Australia']
print(f"Age-specific ratio range on the five-year grid: {r5.min():.3f} ({r5.idxmin()}) "
      f"to {r5.max():.3f} ({r5.idxmax()})")

# Figure 2: full-population standards (band) versus age scopes (stems).
ypos = np.arange(len(scope_ratio))[::-1]
fig, ax = plt.subplots(figsize=(9.0, 3.6), layout='constrained')
ax.axvspan(full_ratio.min(), full_ratio.max(), color='#d8e1d2', alpha=0.5, zorder=0,
           label='Range across five full-population standards')
ax.axvline(1.0, color='#7b8078', linewidth=1, linestyle='--')
for y, (name, value) in zip(ypos, scope_ratio.items()):
    ax.plot([1.0, value], [y, y], color=GREEN, linewidth=2.2, alpha=0.55,
            solid_capstyle='round', zorder=2)
    ax.plot(value, y, 'o', color=GREEN, markersize=7, zorder=3)
    ax.text(value + 0.02, y, f'{value:.2f}', va='center', ha='left', fontsize=9.5, color='#313e38')
ax.set_yticks(ypos)
ax.set_yticklabels(list(scope_ratio.index))
ax.set_xlim(0.9, 2.15)
ax.set_ylim(-0.6, 3.6)
ax.set_xlabel('Malaysia ÷ Australia standardized ratio')
ax.set_title('Which ages count changes the answer', loc='left', pad=14)
ax.legend(loc='lower right', fontsize=8.5)
ax.set_axisbelow(True)
ax.grid(axis='x', color='#e6e8e3', linewidth=0.8)
plt.show()

**Figure 2 | The answer depends on which ages carry the weight.** Shaded band: the 1.459–1.518 range spanned by the five full-population standards. Points: standardized ratios obtained by restricting the WHO standard to one age range and renormalizing its weights within that range. Children are the sharpest divergence (2.01), older ages the mildest (1.41); the all-ages value (1.47) is the WHO number from the table. These are not competing estimates of one quantity. They are answers to different but legitimate questions, and the report must say which one it is answering.

## 4. The Indirect Route's Hidden Standard

Section 1 proved that the SMR is the direct ratio computed on the study population's own structure. Section 2 supplied the mechanism: the SMR's weights are $q_i(x)=p_i(x)\,m_s(x)/\sum_y p_i(y)\,m_s(y)$, the study population's expected deaths under the standard schedule. Change the study population's age structure and $q_i$ changes, even if its age-specific rates do not.

Make this concrete with a controlled comparison. Give two districts the same age-specific mortality: both follow Australia's 1988 schedule multiplied by an age-specific factor that runs from 2.0 below age 30 to 0.6 at ages 65 and over. Give the first district Malaysia's age structure and the second Australia's. The districts now have, by construction, identical mortality at every single age, so a direct comparison on any common standard returns exactly 1.00. Yet their SMRs against the Australian schedule are 1.03 and 0.80.

*The factor is a teaching device, not a claim about Malaysia or Australia. It is chosen to vary with age; if the relative schedule were proportional to the standard's, every weighting would return the same number (Section 2) and the point would vanish.*

Neither figure is a mistake, and neither is noise. The SMR at Australia's structure coincides with the direct standardized ratio on Australia's own structure — Section 1's identity — so the two routes agree only when the study structure happens to be the standard's. At Malaysia's structure the SMR is a different weighted average of the same age-specific ratios. The ratio of the two SMRs, 1.29, is therefore not a mortality difference between the districts. It is a difference between two implicit standards.

The practical rules sharpen the warning at the end of Lesson 1.3a:

- An SMR validly compares **one population with the external standard**. It does not rank that population against another whose age structure differs.
- To compare groups, standardize directly on **one common standard**. If the cell counts are too small for that, report each SMR with its uncertainty and resist fine rankings.
- If two populations have identical age-specific rates but different structures, an SMR difference between them is pure composition — the very thing standardization was supposed to remove.

In [ ]:
# ---- 5. Same age-specific rates, different age structures ----
# A teaching schedule: study mortality relative to Australia, by age.
r_true = pd.Series(
    np.where(mid < 30, 2.0,
             np.where(mid < 50, 1.5,
                      np.where(mid < 65, 1.0, 0.6))),
    index=asdr.index)
m_true = asdr['Australia'] * r_true

# Identical rates for both districts: a direct comparison on one common standard is 1.00.
district_rates = pd.DataFrame({'young district': m_true, 'old district': m_true})
on_common = standardize_direct(district_rates, structure['Australia'])
assert np.isclose(on_common['young district'] / on_common['old district'], 1.0)
direct_std = on_common['young district']
direct_rel = direct_std / cdr['Australia']
print(f"Direct standardized rate on the Australian standard: {direct_rel:.3f} "
      f"(relative to Australia's crude rate), for either district")

share65, smr = [], []
for lam in np.linspace(0, 1, 41):
    mix = (1 - lam) * structure['Malaysia'] + lam * structure['Australia']
    share65.append(float(mix[mid >= 65].sum()) * 100)
    smr.append(float((mix * m_true).sum() / (mix * asdr['Australia']).sum()))
smr = pd.Series(smr, index=share65)

# At Australia's structure the indirect estimate collapses to the direct one (Section 1's identity).
assert np.isclose(smr.iloc[-1], direct_rel)
assert np.isclose(smr.iloc[0], 1.034, atol=0.002)
assert smr.is_monotonic_decreasing
print(f"SMR at Malaysia's structure: {smr.iloc[0]:.3f}; at Australia's structure: {smr.iloc[-1]:.3f}")
print(f"Ratio of the two SMRs: {smr.iloc[0] / smr.iloc[-1]:.3f}; "
      f"direct comparison between the districts: 1.00")

# Figure 3: SMR versus study structure, direct ratio held fixed.
fig, ax = plt.subplots(figsize=(9.0, 3.9), layout='constrained')
ax.plot(smr.index, smr, color=GREEN, linewidth=2.0, label='Indirect: SMR')
ax.axhline(direct_rel, color=RUST, linewidth=1.6, linestyle='--',
           label='Direct on the Australian standard')
ax.plot([smr.index[0], smr.index[-1]], [smr.iloc[0], smr.iloc[-1]], 'o',
        color=GREEN, markersize=5)
ax.annotate(f'{smr.iloc[0]:.2f}', (smr.index[0], smr.iloc[0]), textcoords='offset points',
            xytext=(8, 3), fontsize=9.5, color=GREEN)
ax.annotate(f'{smr.iloc[-1]:.2f}', (smr.index[-1], smr.iloc[-1]), textcoords='offset points',
            xytext=(-34, 8), fontsize=9.5, color=GREEN)
ax.set_xlim(3.2, 11.4)
ax.set_ylim(0.76, 1.06)
ax.set_xlabel('Share of the population aged 65+ (%)')
ax.set_ylabel('Mortality relative to\nthe Australian schedule')
ax.set_title('Same age-specific rates, different age structures', loc='left', pad=14)
ax.legend(loc='upper right', fontsize=9)
ax.set_axisbelow(True)
ax.grid(axis='y', color='#e6e8e3', linewidth=0.8)
ax.text(3.35, 0.775, 'identical age-specific rates at every point on this curve',
        fontsize=8.5, color='#7b8078')
plt.show()

**Figure 3 | The SMR moves with the study population's age structure; the direct ratio does not.** Both districts have identical age-specific rates: Australia's 1988 schedule multiplied by a factor falling from 2.0 at young ages to 0.6 at ages 65 and over. The horizontal axis interpolates between Malaysia's 1988 age structure (left) and Australia's (right). Green: the SMR against the Australian schedule, 1.03 at Malaysia's structure and 0.80 at Australia's. Rust: the direct standardized rate on the Australian standard, 0.80 at every structure, because both the rates and the weights are fixed. The two meet at the right-hand end, where the study structure is the standard (Section 1's identity). The direct comparison between the two districts is 1.00, although their SMRs differ by 29%.

## 5. What Standardization Cannot Fix

Standardization is a reweighting, not an adjustment in the causal sense, and a single ratio is a summary, not a portrait. Five limits follow from the algebra.

1. **It cannot choose the weights for you.** A standardized rate answers a counterfactual question under a specified weighting. When the age-specific ratios vary, different reasonable weightings give different answers — 1.41 to 2.01 across the age scopes in Section 3, before the choice of full-population standard is even considered.
2. **It cannot make differently weighted numbers comparable.** An SMR, a direct standardized rate, and a ratio of two SMRs are three different objects. The 2.21 from Lesson 1.3a is not a larger version of 1.52; it is a different question, computed on two private standards.
3. **It cannot show where the difference lies.** Malaysia's ratio is 3.12 at ages 1–4 and 0.98 at 85+. Neither fact is visible in a single 1.5, and the sign of the comparison is not uniform across ages.
4. **It cannot adjust for variables it omits.** Reweighting by age leaves sex, socioeconomic position, health behaviour, and every other compositional difference in place. A standardized comparison describes two schedules under one weighting; it does not estimate a causal effect.
5. **It cannot manufacture precision.** Lesson 1.3a's Poisson floor still applies: a small population's SMR carries its own sampling uncertainty, and a directly standardized rate built on thin cells is noisier still. A precise-looking ratio can rest on a handful of deaths.

### When to report age-specific rates directly

Standardization is the right tool when the comparative question is genuinely about the whole schedule and the age-specific ratios are reasonably stable. It is the wrong tool when any of the following holds:

- **The question is about specific ages.** "How do the countries compare in childhood?" is not answered by an all-ages ratio; report the 0–14 comparison (2.01 here), or the age-specific rates.
- **The schedules cross.** A single ratio cannot represent a comparison whose sign changes with age. Here the ratio falls below 1 only at 85+, but nothing in the method flags that; only the curve or the table does.
- **The standard is contested or the choice is consequential.** Report the range across standards, not one number, and name the standard behind the headline figure.
- **The counts are large enough for age-specific rates.** If they are, the curve is more informative than any summary. If they are not, say so instead of implying a precision the data lack.
- **The audience needs to see where the burden is.** A policy or clinical audience usually needs the age profile, not only its weighted average.

In practice the honest report often gives both: a summary ratio with its standard named, plus the age-specific curve or a short table. A defensible sentence for the data in this notebook:

> Malaysia's mortality was 1.52 times Australia's under the Malaysian age structure and 1.46 times under the Australian structure; it was about 2.0 times at ages 0–14, 1.41 times at ages 65 and over, and 0.98 at 85+.

Every number in that sentence carries its weights. None is "the" answer, and the sentence is more useful for saying so.

In [ ]:
# ---- 6. Assemble a complete report for the same comparison ----
report = pd.DataFrame({
    'Comparison': [
        'Crude rate ratio (own structures)',
        'Direct, Australian structure',
        'Direct, Malaysian structure',
        'Direct, WHO standard',
        'SMR, Malaysia vs Australian schedule',
        'Ages 0–14, WHO weights',
        'Ages 15–64, WHO weights',
        'Ages 65+, WHO weights',
        'Ages 85+ (crude)',
    ],
    'Malaysia ÷ Australia': [
        cdr['Malaysia'] / cdr['Australia'],
        ratio_on_aus,
        ratio_on_mal,
        full_ratio['WHO'],
        smr_mal,
        scope_ratio['Ages 0–14'],
        scope_ratio['Ages 15–64'],
        scope_ratio['Ages 65+'],
        r_age['85+'],
    ],
}).set_index('Comparison')
display(report.round(3))

assert np.isclose(report.loc['Direct, WHO standard', 'Malaysia ÷ Australia'], scope_ratio['All ages'])
print(f"Malaysia/Australia: {ratio_on_mal:.2f} under Malaysian weights, "
      f"{ratio_on_aus:.2f} under Australian weights, "
      f"{scope_ratio['Ages 0–14']:.2f} at ages 0–14, "
      f"{scope_ratio['Ages 65+']:.2f} at ages 65+, and {r_age['85+']:.2f} at 85+.")

## Check Your Understanding

1. Using the two panels of Figure 1, explain in one sentence why five full-population standards agree (1.46–1.52) even though the age-specific ratio ranges from 0.98 to 3.12.
2. Two districts have SMRs of 1.10 and 0.90 against the same national schedule, yet their age-specific death rates are identical. How is that possible, and what should the report say?
3. A journal asks for "the age-standardized mortality ratio" for Malaysia and Australia with no standard specified. Write the table or figure you would submit, and justify each row or column.

**Answers.** (1) The ratio is nearly flat (about 1.45–1.7) across the ages where every full-population standard places most of its expected deaths, roughly 30–84; the extreme ages (1–4 at 3.12, 85+ at 0.98) carry negligible weight in every full-population standard, so the weighted average is stable. (2) Their age structures differ: each SMR carries its own implicit standard (Section 4). Report the direct standardized comparison on one common standard, or the age-specific rates; an SMR table without each group's structure is misleading. (3) A table that names the standard population in every row, gives at least two standards plus the range, and adds either an age-specific ratio curve or a table showing the 85+ crossover — with the crude ratio for context. Do not report a single "the" standardized ratio.

## Coming Next: From Rates to Probabilities

Module 1 turns next from comparing mortality schedules to constructing the life table. Lesson 1.4a follows a single cohort through life and asks what distinguishes a death rate $m_x$ from a probability of dying $q_x$; the answer is the denominator, and it determines how every later column of the life table is built.

[Back to the Demographic Analysis Series](/blog/series/demographic) · [Review Lesson 1.3a](/blog/demographic-m1l3a-indirect-standardization-smr) · [Review Lesson 1.2b](/blog/demographic-m1l2b-standardization-worked-example)

## References and Data Sources

1. Carmichael, G. A. (2016). *Fundamentals of Demographic Analysis: Concepts, Measures and Methods*. Springer. Data: Ch. 1, Table 1.2 (age-specific death rates, per 1,000) and Table 1.3 (population by age and sex, per 1,000,000), Malaysia and Australia, 1988. Direct and indirect standardization and the Additional Points on the limits of standardization: Ch. 2, pp. 68–72. [Springer book and chapter listing](https://link.springer.com/book/10.1007/978-3-319-23255-3).
2. Wachter, K. W. (2014). *Essential Demographic Methods*. Harvard University Press. Age standardization: §6.5. [Publisher's book page](https://www.hup.harvard.edu/books/9780674045576).
3. Ahmad, O. B., Boschi-Pinto, C., Lopez, A. D., Murray, C. J. L., Lozano, R., & Inoue, M. (2001). *Age Standardization of Rates: A New WHO Standard*. GPE Discussion Paper Series No. 31. World Health Organization. Segi and WHO standard weights: Table 1. [Original WHO PDF](https://cdn.who.int/media/docs/default-source/gho-documents/global-health-estimates/gpe_discussion_paper_series_paper31_2001_age_standardization_rates.pdf).
4. Breslow, N. E., & Day, N. E. (1987). *Statistical Methods in Cancer Research, Volume II: The Design and Analysis of Cohort Studies*. IARC Scientific Publications No. 82. The canonical treatment of SMR variability and of the comparability of SMRs across populations. [IARC publication page](https://publications.iarc.fr/Book-And-Report-Series/Iarc-Scientific-Publications/Statistical-Methods-In-Cancer-Research-Volume-II-The-Design-And-Analysis-Of-Cohort-Studies-1986).

**Reproducibility.** All data are embedded in the code cells, so execution requires no external downloads. Dependencies are NumPy, pandas, Matplotlib, and Jupyter/IPython. The computations are deterministic; no random seed is needed. Assertions check the crude rates (4.95 and 7.25 per 1,000), the two standardization functions on toy tables, the SMR–direct identity in both directions, the identity $R=\sum_x q(x)r(x)$ for both own structures, the five-standard band, the four age-scope ratios, the 85+ ratio, and the structure-driven SMR gap (1.03 versus 0.80) with the direct comparison fixed at 1.00.